# Exploratory Data Analysis

In [267]:
# load packages
import pandas as pd
import numpy as np
import glob


In [268]:
# get list of file paths
file_paths = glob.glob("./Data/raw/citi_costco*.CSV")

# loop through each file in the folder and essentially union all the csvs together into one dataframe
citi_costco_transactions = pd.concat([pd.read_csv(file) for file in file_paths], ignore_index=True)

citi_costco_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Status       98 non-null     object 
 1   Date         98 non-null     object 
 2   Description  98 non-null     object 
 3   Debit        95 non-null     float64
 4   Credit       3 non-null      float64
 5   Member Name  96 non-null     object 
dtypes: float64(2), object(4)
memory usage: 4.7+ KB


### Extract
Lets figure out how to extract the data from the file

In [269]:
from dataclasses import dataclass
from pathlib import Path
from __future__ import annotations

@dataclass
class RawStatement:
    """Container for one raw CSV plus metadata about where it came from."""
    source_name: str          # e.g. "chase_credit_card" — matches config.yaml key
    file_path: Path
    df: pd.DataFrame

In [270]:
# Load the configuration
import yaml

def load_config(config_path: str = "./config.yaml") -> dict:
    """Load the source/column-mapping config."""
    # TODO: open config_path, yaml.safe_load(), return dict
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

config__ = load_config()
print(config__["sources"].items())

dict_items([('citi_costco_credit_card', {'file_pattern': 'citi_costco*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('amex_gold_credit_card', {'file_pattern': 'amex_gold*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_credit_card', {'file_pattern': 'iq_credit_card*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_checking', {'file_pattern': 'iq_checking*.csv', 'column_map': {'month': None, 'date': 'Posting Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'},

In [271]:
# this function will find all files in the raw_data_dir that match the file_pattern for a given source
def find_files_for_source(raw_data_dir: Path, file_pattern: str) -> list[Path]:
    """Glob raw_data_dir for files matching this source's pattern."""
    # TODO: return sorted list of Path objects matching file_pattern
    case_insensitive_pattern = file_pattern.replace(".csv", ".[cC][sS][vV]")
    file_paths = glob.glob(f"{raw_data_dir}/{case_insensitive_pattern}")
    return sorted(Path(f) for f in file_paths)

root_data_dir = "./Data/raw"
citi_files = find_files_for_source(root_data_dir, "citi_costco*.CSV")
amex_files = find_files_for_source(root_data_dir, "amex_gold_credit_card*.CSV")
iq_files = find_files_for_source(root_data_dir, "iq_credit_card*.CSV")

print("Citi files:", citi_files)
print("Amex files:", amex_files)
print("IQ files:", iq_files)


Citi files: [PosixPath('Data/raw/citi_costco_2026-06-19.CSV'), PosixPath('Data/raw/citi_costco_2026-07-21.CSV')]
Amex files: []
IQ files: []


In [272]:
def extract_all(config: dict) -> list[RawStatement]:
    """
    Main entry point for this module.
    Loop through every source in config['sources'], find matching files,
    read each into a DataFrame, and return a list of RawStatement objects.
    """
    raw_statements: list[RawStatement] = []
    root_data_dir = "./Data/raw"

    # TODO:
    for source_name, source_cfg in config["sources"].items():
        files = find_files_for_source(root_data_dir, source_cfg["file_pattern"])
        for f in files:
            df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
            raw_statements.append(RawStatement(source_name, f, df))

    return raw_statements

if __name__ == "__main__":
    # Quick manual test when running this file directly
    cfg = load_config()
    statements = extract_all(cfg)
    print(f"Extracted { len(statements)} raw statement(s).")


Extracted 2 raw statement(s).


### Transform
Now lets transform the raw statements we pull from the extract portion. We'll want to normalize column names and data types across all sources.

    txn_id | month | date | category | subcategory | amount | description | source

`amount` convention: negative = money out (expense), positive = money in.
This is the "canonical" sign convention for the whole pipeline — resolve
each source's raw sign convention against this in normalize_amount_sign().

In [273]:
# transform setup
from datetime import datetime
import hashlib

import pandas as pd

from extract import RawStatement # import RawStatement from extract.py

NORMALIZED_COLUMNS = ["txn_id", "month", "date", "category", "subcategory", "amount", "description", "source"]

In [274]:
# rename columns
statements = extract_all(config__)

def rename_columns(df: pd.DataFrame, column_map: dict) -> pd.DataFrame:
    """
    Rename raw columns to normalized names using column_map from config.
    TODO: invert column_map (normalized -> raw) into (raw -> normalized)
          and call df.rename(columns=...). Handle missing/null mappings
          (e.g. category: null) by creating an empty column instead.
    """
    df = df.copy()
    for normalized_name, raw_name in column_map.items():
        if raw_name is None:
            df[normalized_name] = np.nan  # Create an empty column if mapping is None
        else:
            df = df.rename(columns={raw_name: normalized_name})
    return df

# for statement in statements:
#     source_name = statement.source_name
#     df = statement.df
#     column_map = config__["sources"][source_name]["column_map"]
#     df_renamed = rename_columns(df, column_map)
#     print(f"Renamed columns for {source_name}:")
#     print(df_renamed.head())


In [275]:
# normalize dates
def normalize_dates(df: pd.DataFrame, date_format: str) -> pd.DataFrame:
    """Parse the date column using the source's date_format into pd.Timestamp."""
    # TODO: pd.to_datetime(df["date"], format=date_format, errors="coerce")
    # TODO: log/flag any rows that failed to parse (errors="coerce" -> NaT)
    df["date"] = pd.to_datetime(df["date"], format=date_format, errors="coerce")
    return df

# for statement in statements:
#     source_name = statement.source_name
#     df = statement.df
#     column_map = config__["sources"][source_name]["column_map"]
#     df_renamed = rename_columns(df, column_map)
#     # print(f"Renamed columns for {source_name}:")
#     # print(df_renamed.head())
#     date_format = config__["sources"][source_name]["date_format"]
#     df_normalized = normalize_dates(df_renamed, date_format)
#     print(f"Normalized dates for {source_name}:")
#     print(df_normalized.head())

In [276]:
# normalize amount sign
def normalize_amount_sign(df: pd.DataFrame, amount_sign: str) -> pd.DataFrame:
    """
    Flip sign if needed so the canonical convention holds:
    negative = expense, positive = income/refund.
    """
    if amount_sign == "positive_is_expense":
        df["amount"] = df["amount"]
    return df

# for statement in statements:
#     source_name = statement.source_name
#     df = statement.df
#     column_map = config__["sources"][source_name]["column_map"]
#     df_renamed = rename_columns(df, column_map)
#     date_format = config__["sources"][source_name]["date_format"]
#     df_normalized = normalize_dates(df_renamed, date_format)
#     amount_sign = config__["sources"][source_name]["amount_sign"]
#     df_final = normalize_amount_sign(df_normalized, amount_sign)
#     print(f"Normalized amount sign for {source_name}:")
#     print(df_final.head())

In [277]:
# add transaction id
def add_txn_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a stable hash ID per transaction (date+description+amount+source)
    so we can dedupe transactions that show up in multiple exports
    (e.g. re-downloading overlapping date ranges).
    """
    # TODO: df["txn_id"] = df.apply(lambda r: hashlib.md5(
    #     f"{r['date']}{r['description']}{r['amount']}{r['source']}".encode()
    # ).hexdigest(), axis=1)
    df["txn_id"] = df.apply(lambda r: hashlib.md5(
        f"{r['date']}{r['description']}{r['amount']}{r['source']}".encode()
    ).hexdigest(), axis=1)
    return df

# for statement in statements:
#     source_name = statement.source_name
#     df = statement.df
#     column_map = config__["sources"][source_name]["column_map"]
#     amount_sign = config__["sources"][source_name]["amount_sign"]
#     date_format = config__["sources"][source_name]["date_format"]
#     df_renamed = rename_columns(df, column_map)
#     df_normalized = normalize_dates(df_renamed, date_format)
#     df_final = normalize_amount_sign(df_normalized, amount_sign)
#     df_final["source"] = source_name  # Add source column for txn_id generation
#     df_final["month"] = df_final["date"].dt.to_period("M")  # Add month column for grouping
#     df_final = add_txn_id(df_final)
#     df_final = df_final.dropna(subset=['amount']) # Drop rows where amount is NaN
#     df_final = df_final[NORMALIZED_COLUMNS]  # Keep only normalized columns
#     print(f"Normalized {source_name}:")
#     print(df_final.to_string(index=False, max_rows=100))  # Print first 5 rows without index

In [278]:
# catgorize
# Define category mapping rules based on keywords
subcategory_map = {
    "Groceries": ["supermarket", "walmart", "trader joe", "kroger", "costco whse", "whole foods", "h mart", "h-mart", "hmart"],
    "Dining": ["restaurant", "starbucks", "mcdonalds", "cafe", "chipotle", "dunkin", "burger king", "subway"],
    "Utilities": ["electric", "water", "internet", "gas & electric", "xfinity", "verizon", "spectrum"],
    "Transport": ["uber", "lyft", "shell", "chevron", "transit", "costco gas", "orca"],
    "Shopping": ["amazon", "target", "best buy", "walmart", "home depot", "lowes", "REI", "nike", "adidas", "zara", "h&m", "abercrombie", "gap", "old navy", "macys", "nordstrom", "sephora", "ulta"],
    "Subscriptions": ["netflix", "spotify", "hulu", "disney+", "peacock", "adobe", "dropbox", "ZWIFT", "peloton", "apple music", "prime video", "audible", "strava", "amazon web services", "aws", "google drive", "icloud", "microsoft office", "office 365"],
    "Travel": ["airbnb", "delta", "united", "southwest", "expedia", "hotels.com", "alaska air", "american airlines", "united airlines", "jetblue", "travelocity", "kayak"],
}

category_map = {
    "Essentials": ["Groceries", "Utilities", "Transport"],
    "Non-Essentials": ["Dining", "Shopping", "Subscriptions", "Travel"],
    "Investments": ["Savings", "Investments", "Retirement"],
}

def subcategorize(desc: str) -> str:
    """
    Fill in missing categories using simple keyword rules on `description`
    (e.g. "STARBUCKS" -> "Coffee", "SHELL OIL" -> "Gas").
    TODO: start with a small dict of {keyword: category}, expand over time.
    Leave category as-is if the source already provided one.
    """
    desc_lower = str(desc).lower()
    for category, keywords in subcategory_map.items():
        for keyword in keywords:
            if keyword in desc_lower:
                return category
    return "Other"  # Default category if no keywords match

def categorize(subcategory: str) -> str:
    """
    Fill in missing categories using simple keyword rules on `description`
    (e.g. "STARBUCKS" -> "Coffee", "SHELL OIL" -> "Gas").
    TODO: start with a small dict of {keyword: category}, expand over time.
    Leave category as-is if the source already provided one.
    """
    sub_cat_lower = str(subcategory).lower()
    for category, keywords in category_map.items():
        for keyword in keywords:
            if keyword in sub_cat_lower:
                return category
    return "Other"  # Default category if no keywords match



In [279]:
# call all fxns to normalize statement
def normalize_statement(stmt: RawStatement, source_cfg: dict) -> pd.DataFrame:
    """Run one RawStatement through the full normalization pipeline."""
    df = stmt.df.copy()
    df = rename_columns(df, source_cfg["column_map"])
    df = normalize_dates(df, source_cfg["date_format"])
    df = normalize_amount_sign(df, source_cfg["amount_sign"])
    df["source"] = stmt.source_name
    df["month"] = df["date"].dt.to_period("M")  # Add month column for grouping
    df = add_txn_id(df)
    df["subcategory"] = df["description"].apply(subcategorize)
    df["category"] = df["subcategory"].apply(categorize)
    df = df.dropna(subset=['amount']) # Drop rows where amount is NaN 
    return df[NORMALIZED_COLUMNS]



In [280]:
def transform_all(raw_statements: list, config: dict) -> pd.DataFrame:
    """
    Normalize every RawStatement, concatenate into one master DataFrame,
    and dedupe on txn_id.
    """
    normalized_frames = []

    # TODO:
    for stmt in raw_statements:
        source_cfg = config["sources"][stmt.source_name]
        normalized_frames.append(normalize_statement(stmt, source_cfg))

    combined = pd.concat(normalized_frames, ignore_index=True)
    combined = combined.drop_duplicates(subset="txn_id")
    combined = combined.sort_values("date")

    return combined

if __name__ == "__main__":
    from extract import load_config, extract_all

    cfg = load_config()
    raw = extract_all(cfg)
    clean = transform_all(raw, cfg)
   # Force pandas to print all columns and format nicely
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.colheader_justify', 'center')

    print(clean.head(10))  # Pint first 10 rows without index
    print(f"Total transactions after deduplication: {len(clean)}")

                 txn_id               month      date    category  subcategory    amount                     description                              source         
29  a0391f068559ac76c716515683173d38  2026-05 2026-05-23   Other   Subscriptions    16.25                  Audible*MO5K997K3 Amzn.com/billNJ  citi_costco_credit_card
27  2c8197f7af32571c0312fcac15373c6d  2026-05 2026-05-24   Other       Transport     3.00                         ORCA*00RSFPF 2063985346 WA  citi_costco_credit_card
28  cbb7770de040cff6417127c405a7490b  2026-05 2026-05-24   Other   Subscriptions    87.11                 Strava Subscription 415-4631132 CA  citi_costco_credit_card
26  f8eebc49bebab881ee293bde87560fe9  2026-05 2026-05-29   Other           Other    18.88                           SQ *BASILISK Portland OR  citi_costco_credit_card
25  3e872bbc347c0ef998eb4f69a44ebafd  2026-05 2026-05-29   Other       Groceries    17.51                         H MART BELMONT PORTLAND OR  citi_costco_credit_card
24  

### Load